# Nhóm 7, Notebook_B (Sinh viên B): chia dữ liệu, năm mô hình, tinh chỉnh, RQ3, SHAP, chẩn đoán

**Đề tài**: Sự cố UAS đa phương thức và tóm tắt nhanh (NASA ASRS)

**Khung SDG**: SDG 9: hạ tầng hàng không an toàn, "the industrial internet".

**Vai trò**: Sinh viên B nhận `data/processed/feat.parquet` và `manifest.json` từ Notebook_A, làm Bước 4 và Bước 6, vẽ hình so sánh mô hình, giải thích chỉ số, tinh chỉnh, câu hỏi thứ ba, SHAP và chẩn đoán vì sao tốt hay chưa tốt. Mọi bảng lưu `report/table_*.csv`, mọi hình `report/fig_rq2_*.png`, `fig_rq3_*.png`.

**Quy tắc**: không chạm tập kiểm thử khi tinh chỉnh; mỗi kết luận phải kèm số; báo cáo slot 1 thứ Ba và thứ Sáu.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: kiểm tra bàn giao từ Notebook_A

Đọc `manifest.json`, so mã băm và số dòng với file parquet; nếu lệch thì báo Sinh viên A chạy lại Notebook_A trước khi tiếp tục.

In [1]:
import pandas as pd, numpy as np, json, hashlib, duckdb, warnings, time
from pathlib import Path
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); np.random.seed(42)
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE, f1_score, average_precision_score, roc_auc_score
current = Path.cwd()
project_root = current
while project_root.name and project_root.name != "Group6_ADY201m_FA26_AI2108":
    if (project_root / "ASRS_FULL_DATASET.csv").exists():
        break
    project_root = project_root.parent

man = json.load(open(project_root / 'data/processed/manifest.json'))
h = hashlib.md5(open(project_root / 'data/processed/feat.parquet', 'rb').read()).hexdigest()
assert h == man['md5'], 'parquet differs from the file handed over by Student A'
feat = pd.read_parquet(project_root / 'data/processed/feat.parquet')
assert len(feat) == man['rows'], 'row count differs from manifest'
con = duckdb.connect()
con.register('feat', feat)          # expose the handed-over table to SQL in this notebook
print('handover OK:', man['rows'], 'rows,', len(man['cols']), 'columns, from', man['time_min'], 'to', man['time_max'])
def style(ax): ax.spines[['top', 'right']].set_visible(False)

handover OK: 110988 rows, 6 columns, from 2002 to 2025


## Bước 4: chia dữ liệu và bảng đặc trưng

**Làm gì**: chọn `X_cols` từ `report/table_columns.csv` của A; chia theo thời gian (hoặc theo đơn vị với leave-out); tính mốc naive; xuất bảng đặc trưng (tên, cách tính, nhóm) cho bài.

**Vì sao chia theo thời gian**: dữ liệu tương lai không được rò vào huấn luyện; chia ngẫu nhiên cho kết quả đẹp giả (Bergmeir và Benitez, 2012).

**Kiểm tra**: mọi thời điểm của test sau train; số dòng test đủ lớn (trên 10% và trên 3 nghìn dòng).

In [2]:
# Step 4: three representations, split by year
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
feat = con.execute("SELECT * FROM feat").df()   
train = feat[feat.year <= 2023]; test = feat[feat.year >= 2024]
struct_cols = ['flight_phase','weather','light','altitude','airspace','mission','aircraft_type','is_uas']
Xs_tr = pd.get_dummies(train[struct_cols].astype(str)); Xs_te = pd.get_dummies(test[struct_cols].astype(str)).reindex(columns=Xs_tr.columns, fill_value=0)
tf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
Xt_tr = tf.fit_transform(train.narrative); Xt_te = tf.transform(test.narrative)
Xc_tr = hstack([csr_matrix(Xs_tr.values), Xt_tr]); Xc_te = hstack([csr_matrix(Xs_te.values), Xt_te])
y_tr, y_te = train.y, test.y     # metric: f1_score(average='macro')
X_tr, X_te = Xs_tr, Xs_te        # structured representation for the shared model loop; text and combined below
X_cols = list(Xs_tr.columns)
# combined representation: fit the same models on Xc_tr / Xc_te and add rows 'LightGBM combo', 'Logistic TF-IDF' to the results table
feat_table = pd.DataFrame({'feature': X_cols, 'group': ['lag/rolling' if any(k in c for k in ['lag', 'ma', 'max', 'rain', 'sum']) else ('calendar' if c in ('dow', 'mon', 'hr', 'doy', 'MONTH', 'HOUR', 'DAY_WEEK') else 'exogenous/structural') for c in X_cols]})
feat_table.to_csv('report/table_features.csv', index=False); print(feat_table)
assert len(X_te) > max(3000, 0.1 * len(X_tr)), 'test set too small'

KeyError: "['weather', 'light', 'altitude', 'airspace', 'mission', 'aircraft_type'] not in index"

### 6.0. Giải thích các chỉ số trước khi chạy

| Chỉ số | Công thức | Dùng khi | Đọc thế nào | Bẫy thường gặp |
|---|---|---|---|---|
| MAE | trung bình của |y − ŷ| | hồi quy, đơn vị như mục tiêu | càng nhỏ càng tốt; so với mốc naive | không phạt lỗi lớn |
| RMSE | căn của trung bình (y − ŷ)² | khi lỗi lớn quan trọng (lũ, đỉnh tải) | lớn hơn MAE nếu có lỗi lớn | nhạy ngoại lai |
| MAPE | trung bình của |y − ŷ| / |y| | so sánh giữa vùng có quy mô khác nhau | phần trăm | nổ khi y gần 0 |
| nRMSE | RMSE chia công suất định mức | điện mặt trời, gió | phần trăm định mức | cần cùng định mức |
| NSE | 1 − Σ(y − ŷ)² / Σ(y − ȳ)² | thuỷ văn | 1 là hoàn hảo, 0 bằng trung bình, âm tệ hơn trung bình | bị chi phối bởi đỉnh |
| Accuracy | đúng / tổng | lớp cân bằng | vô nghĩa khi lớp ít mẫu | 95% accuracy với 5% dương |
| Precision, Recall | TP/(TP+FP), TP/(TP+FN) | cảnh báo, lớp ít mẫu | recall là không bỏ sót; precision là ít báo giả | phụ thuộc ngưỡng |
| F1, macro-F1 | trung bình điều hoà; macro là trung bình các lớp | nhiều lớp lệch | mọi lớp có trọng số bằng nhau | che giấu lớp nào yếu |
| PR-AUC | diện tích đường precision-recall | lớp dương hiếm | không phụ thuộc ngưỡng | so với tỷ lệ dương |
| ROC-AUC | diện tích đường ROC | xếp hạng | 0,5 là ngẫu nhiên | lạc quan khi lớp lệch |
| ECE | Σ |acc − conf| có trọng số theo bin | cần xác suất tin được | càng nhỏ càng tốt | phụ thuộc số bin |
| Coverage | tỷ lệ y nằm trong khoảng | conformal | phải gần 1 − α | khoảng quá rộng vô dụng |

**Nguyên tắc so sánh**: cùng train/test, cùng đặc trưng, năm seed; báo trung bình ± độ lệch; chênh nhỏ hơn độ lệch thì chưa kết luận được, cần kiểm định Wilcoxon theo cặp; luôn kèm cột thời gian huấn luyện.

## Bước 6a: năm mô hình, năm seed, cùng train/test

**Làm gì**: chạy vòng lặp; thêm hàng mốc naive (từ Bước 4) và hàng baseline chạy lại vào bảng; lưu `table_rq2.csv`.

**Cách chọn năm mô hình**: một mốc tuyến tính (ridge hoặc logistic), một rừng ngẫu nhiên, hai cây tăng cường (LightGBM, XGBoost hoặc CatBoost), một đối chứng theo hướng nghiên cứu (mô hình nền, TabPFN, GRU nhỏ, DistilBERT). Lý do: mốc để biết cải thiện thật, tuyến tính để có hệ số, cây để bắt phi tuyến với chi phí thấp, đối chứng để trả lời RQ3.

In [ ]:
# Step 6a (classification): loop over five models and five seeds
import numpy as np, pandas as pd, time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder(); ytr = le.fit_transform(y_tr); yte = le.transform(y_te)
Xtr = pd.get_dummies(X_tr.astype(str) if X_tr.select_dtypes(exclude='number').shape[1] == X_tr.shape[1] else X_tr, dummy_na=True)
Xte = pd.get_dummies(X_te.astype(str) if X_te.select_dtypes(exclude='number').shape[1] == X_te.shape[1] else X_te, dummy_na=True).reindex(columns=Xtr.columns, fill_value=0)
Xtr, Xte = Xtr.fillna(Xtr.median(numeric_only=True)), Xte.fillna(Xtr.median(numeric_only=True))
multi = len(le.classes_) > 2
models = {
    'Logistic':      lambda s: LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': lambda s: RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=s),
    'XGBoost':       lambda s: XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=s, eval_metric='logloss'),
    'LightGBM':      lambda s: LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=31, class_weight='balanced', random_state=s, verbose=-1),
    'CatBoost':      lambda s: CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, random_seed=s, verbose=0, auto_class_weights='Balanced'),
}
rows = []
for name, make in models.items():
    f1s, prs, aucs, secs = [], [], [], []
    for seed in range(5):
        m = make(seed); t0 = time.perf_counter(); m.fit(Xtr, ytr); secs.append(time.perf_counter() - t0)
        pred = m.predict(Xte); proba = m.predict_proba(Xte)
        f1s.append(f1_score(yte, pred, average='macro'))
        prs.append(average_precision_score(yte, proba[:, 1]) if not multi else np.nan)
        aucs.append(roc_auc_score(yte, proba[:, 1]) if not multi else roc_auc_score(yte, proba, multi_class='ovr'))
    rows.append([name, np.mean(f1s), np.std(f1s), np.nanmean(prs), np.mean(aucs), np.mean(secs)])
res = pd.DataFrame(rows, columns=['model', 'macroF1', 'sd', 'PR_AUC', 'ROC_AUC', 'train_s'])
res.to_csv('report/table_rq2.csv', index=False); print(res.round(3))
best = models['LightGBM'](0).fit(Xtr, ytr); X_te_enc = Xte; X_tr_enc = Xtr

### 6a.1. Hình so sánh mô hình

**Vì sao cần hình sau khi đã có bảng**: bảng cho con số, hình cho thấy mô hình sai ở đâu (đỉnh hay nền, đơn vị nào, thời kỳ nào), là đầu vào trực tiếp cho phần Discussion và cho hướng cải thiện. Bốn hình: (1) cột metric theo mô hình, cột nhấn là tốt nhất, độ lệch ghi bằng số; (2) dự báo so thực tế trên một đơn vị trong tập kiểm thử; (3) phân phối residual (hồi quy) hoặc ma trận nhầm lẫn (phân loại); (4) sai số theo đơn vị hoặc theo thời kỳ để thấy mô hình yếu ở đâu.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
pred = best.predict(X_te_enc); proba = best.predict_proba(X_te_enc)
fig, ax = plt.subplots(figsize=(7, 3.2)); vals = res.iloc[:, 1].values
bars = ax.bar(res.model, vals, color=['#F4A261' if v == vals.max() else '#1B6B6D' for v in vals]); ax.bar_label(bars, fmt='%.3f', fontweight='bold', fontsize=9); ax.set_ylabel(res.columns[1]); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq2_models.png', dpi=300); plt.close(fig)
fig, ax = plt.subplots(figsize=(5, 4)); ConfusionMatrixDisplay(confusion_matrix(yte, pred), display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Blues'); plt.xticks(rotation=45)
fig.tight_layout(); fig.savefig('report/fig_rq2_confusion.png', dpi=300); plt.close(fig)
from sklearn.metrics import precision_recall_curve
if not multi:
    pr, rc, th = precision_recall_curve(yte, proba[:, 1])
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.plot(rc, pr, color='#1B6B6D'); ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq2_pr_curve.png', dpi=300); plt.close(fig)
per_cls = pd.DataFrame({'class': le.classes_, 'f1': f1_score(yte, pred, average=None)}); per_cls.to_csv('report/table_rq2_per_class.csv', index=False); print(per_cls)
t = test.assign(pred=le.inverse_transform(pred)); err_unit = t.groupby('is_uas').apply(lambda g: f1_score(g['y'].astype(str), g.pred.astype(str), average='macro')).sort_values()
err_unit.round(3).to_csv('report/table_rq2_f1_by_unit.csv'); print('macro-F1 by unit, weakest:', err_unit.head(3).round(3).to_dict())

### 6a.2. Tinh chỉnh tham số: có cần không và làm thế nào

**Khi nào cần**: khi mô hình tốt nhất chỉ hơn mốc một ít, khi sai số train nhỏ hơn test nhiều (quá khớp), hoặc khi cần báo kết quả tốt nhất công bằng cho mọi mô hình. Không tinh chỉnh trên tập kiểm thử.

**Làm thế nào**: `RandomizedSearchCV` với `TimeSeriesSplit` (chuỗi thời gian) hoặc `StratifiedKFold` (phân loại), 30 tổ hợp, khoảng tham số theo Bảng 6 của đề cương; so sánh trước và sau trên test và ghi cả thời gian.

**Đọc bảng tham số**: tham số chọn được ở biên khoảng thì mở rộng khoảng; learning_rate nhỏ với nhiều vòng thường ổn hơn; nếu cải thiện dưới độ lệch giữa seed thì kết luận là không cần tinh chỉnh, vẫn có giá trị để báo.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform, randint, uniform
space = {'learning_rate': loguniform(0.01, 0.1), 'num_leaves': randint(15, 64), 'n_estimators': randint(300, 1200),
         'subsample': uniform(0.6, 0.4), 'colsample_bytree': uniform(0.6, 0.4), 'min_child_samples': randint(10, 100)}
search = RandomizedSearchCV(LGBMClassifier(random_state=0, class_weight='balanced', verbose=-1), space, n_iter=30,
                            cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='f1_macro', random_state=42, n_jobs=-1).fit(X_tr_enc, ytr)
tuned = search.best_estimator_; pred_t = tuned.predict(X_te_enc)
tune = pd.DataFrame([['LightGBM default', f1_score(yte, pred, average='macro')], ['LightGBM tuned', f1_score(yte, pred_t, average='macro')]], columns=['model', 'macro_F1'])
tune.round(3).to_csv('report/table_tuning.csv', index=False); print(tune.round(3)); print('selected parameters:', search.best_params_)
sd_seed = res.loc[res.model == 'LightGBM', 'sd'].values[0]
print('Tuning needed?', 'YES' if (f1_score(yte, pred_t, average='macro') - f1_score(yte, pred, average='macro')) > sd_seed else 'NO, gain is within seed spread')
json.dump(search.best_params_, open('report/params_tuned.json', 'w'), indent=2, default=float)

## Bước 6b: câu hỏi thứ ba của đề

**Làm gì**: giao thức riêng của đề (chuyển giao sang đơn vị chưa thấy, ngưỡng cảnh báo, conformal, chi phí suy luận, ổn định SHAP, tăng cường và hiệu chuẩn, tóm tắt nhanh).

**Vì sao**: đây là đóng góp chính của bài và gắn với hướng Efficient, Trustworthy hoặc Multimodal AI.

**Kiểm tra**: ra một bảng và một hình có số; ghi điều kiện áp dụng (vùng, năm, đơn vị).

In [ ]:
from lightgbm import LGBMRegressor, LGBMClassifier
X_tr, X_te, y_tr, y_te = X_tr_enc, X_te_enc, ytr, yte
# Step 6b: RQ3, fast summarisation with a small model and speculative decoding; measure quality and fabrication
import torch, time, re
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
big = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', torch_dtype=torch.float32)
draft = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', torch_dtype=torch.float32)
def summarise(text, assistant=None):
    msgs = [{'role': 'user', 'content': 'Summarise this aviation safety report in 3 sentences. Use only facts from the report.\n' + text[:3000]}]
    ids = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True)
    t0 = time.perf_counter(); out = big.generate(ids, max_new_tokens=120, assistant_model=assistant, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), time.perf_counter() - t0
uas = test[test.is_uas].sample(300, random_state=0)
def unsupported(summary, source):     # entities (numbers, codes, capitalised names) absent from the source
    ents = set(re.findall(r'\b[A-Z][A-Za-z0-9]{2,}\b|\b\d{2,}\b', summary))
    return len([e for e in ents if e not in source]) / max(len(ents), 1)
rows = [(summarise(t)[1], summarise(t, draft)[1]) for t in uas.narrative[:50]]
print('seconds per summary: standard', np.mean([r[0] for r in rows]).round(2), 'speculative', np.mean([r[1] for r in rows]).round(2))
# ROUGE-L via rouge_score against 100 reference summaries; LLM judge scores 1-5 through an API

## Bước 6c: giải thích bằng SHAP, phụ thuộc từng phần và bảng tham số

**Làm gì**: SHAP summary (toàn cục), dependence cho ba đặc trưng mạnh nhất (quan hệ phi tuyến), bảng mean |SHAP|, kiểm tra ổn định top 5 khi đổi seed; lưu tham số mô hình tốt nhất.

**Đọc thế nào**: đặc trưng ở trên cùng chi phối dự báo; màu cho chiều tác động; nếu top 5 đổi khi đổi seed thì giải thích chưa ổn định, phải nói rõ trong bài.

In [ ]:
X_te = X_te_enc; X_cols = list(X_te_enc.columns)
# Step 6c: SHAP explanations and the selected parameter table
import shap, json
best = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                     subsample=0.8, colsample_bytree=0.8, random_state=0, verbose=-1).fit(X_tr, y_tr)
expl = shap.TreeExplainer(best); sv = expl.shap_values(X_te)
imp = (pd.DataFrame({'feature': X_cols, 'mean_abs_shap': np.abs(sv).mean(0)})
         .sort_values('mean_abs_shap', ascending=False))
imp.to_csv('report/table_shap.csv', index=False)
shap.summary_plot(sv, X_te, show=False); plt.savefig('report/fig_shap.png', dpi=300, bbox_inches='tight')
json.dump(best.get_params(), open('report/params_best.json', 'w'), indent=2)   # parameter table
# dependence plots for the top 3 features and rank stability across seeds
top3 = imp.feature.head(3).tolist()
for f in top3:
    shap.dependence_plot(f, sv, X_te, show=False); plt.savefig(f'report/fig_shap_dep_{f}.png', dpi=300, bbox_inches='tight'); plt.close()
ranks = []
for s in range(3):
    m = type(best)(**{**best.get_params(), 'random_state': s}).fit(X_tr, y_tr)
    v = shap.TreeExplainer(m).shap_values(X_te); v = v[1] if isinstance(v, list) else v
    ranks.append(set(pd.Series(np.abs(v).mean(0), index=X_cols).nlargest(5).index))
print('top-5 overlap across seeds:', len(ranks[0] & ranks[1] & ranks[2]), '/ 5')

## Chẩn đoán: vì sao tốt, vì sao chưa tốt, và hướng cải thiện

Điền vào bảng dưới (lưu `report/table_diagnosis.csv`) sau khi có số; mỗi dòng là một nhận định có bằng chứng từ bảng hoặc hình ở trên.

| Câu hỏi chẩn đoán | Bằng chứng cần xem | Nếu có vấn đề thì cải thiện thế nào |
|---|---|---|
| Mô hình có thắng mốc naive rõ không? | `table_rq2.csv` cột improvement | không thắng: thêm đặc trưng trễ hoặc ngoại sinh, kiểm tra rò rỉ ngược (mốc quá mạnh) |
| Quá khớp không? | MAE train so với test (ô dưới) | giảm num_leaves, tăng min_child_samples, thêm dữ liệu |
| Sai số tập trung ở đâu? | `table_rq2_error_by_unit.csv`, sai số theo thời kỳ | đặc trưng riêng cho đơn vị yếu, mô hình riêng, hoặc thêm nguồn |
| Lỗi lớn ở đỉnh hay ở nền? | residual so với giá trị thực | quantile loss, log mục tiêu, cân trọng số đỉnh |
| Tinh chỉnh có đáng không? | `table_tuning.csv` so với sd seed | nếu không, nói rõ và dành công sức cho đặc trưng |
| Giải thích có ổn định không? | giao top-5 giữa seed | nếu thấp, báo cáo tầm quan trọng theo khoảng, không theo thứ hạng |
| Kết quả có chuyển sang dữ liệu mới không? | bảng RQ3 | tái hiệu chỉnh, đặc trưng bất biến, ghi giới hạn |

Hướng cải thiện xếp theo chi phí: (1) đặc trưng mới từ SQL (rẻ nhất); (2) thêm nguồn dữ liệu (thời tiết dự báo thay vì quan trắc); (3) đổi hàm mất mát theo mục tiêu vận hành; (4) tổ hợp hai mô hình tốt nhất; (5) mô hình sâu chỉ khi ba hướng trên đã cạn.

In [ ]:
tr_f1, te_f1 = f1_score(ytr, best.predict(X_tr_enc), average='macro'), f1_score(yte, pred, average='macro')
diag = pd.DataFrame([['overfitting', f'train {tr_f1:.3f} vs test {te_f1:.3f}', 'signs present' if tr_f1 - te_f1 > 0.1 else 'no'],
                     ['weakest class', per_cls.sort_values('f1').iloc[0].to_dict(), 'revisit class_weight, merge classes, class-specific features']], columns=['question', 'evidence', 'assessment'])
diag.to_csv('report/table_diagnosis.csv', index=False); print(diag)

## Test case của Notebook_B

Chạy trước khi báo cáo; mọi assert phải qua.

In [ ]:
assert not any(str(c).startswith('y') and c != 'y' for c in X_cols), 'features contain the label'
assert f1_score(yte, pred, average='macro') > f1_score(yte, np.full_like(yte, pd.Series(ytr).mode()[0]), average='macro'), 'does not beat the majority-class baseline'
print('Tests B: OK')

## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [ ] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [ ] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [ ] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [ ] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [ ] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [ ] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [ ] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [ ] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [ ] Audit Log cá nhân có đủ prompt của tuần
- [ ] Báo cáo năm mục đã dán vào kênh nhóm trước 7:00 thứ Ba hoặc thứ Sáu


## Mẫu báo cáo tiến độ (slot 1 thứ Ba và thứ Sáu, 7:00 đến 9:15)

Dán vào kênh nhóm trước 7:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................